<!--
---
PURPOSE: Setup configuration, validate dependencies, and create folder structure.
REQUIRES: []
PRODUCES:
  - outputs/reports/config_snapshot.json
WHAT_NEXT: notebooks/01_Session_Discovery_and_Metadata.ipynb
---
-->

# 00 Setup and Configuration

**Purpose**
Setup configuration, validate dependencies, and create folder structure.

**Requires**
- None

**Produces**
- `outputs/reports/config_snapshot.json`

**What to run next**
- `notebooks/01_Session_Discovery_and_Metadata.ipynb`

This notebook configures paths, validates dependencies, and writes a configuration snapshot for reproducibility.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
# Works whether JupyterLab is launched from repo root or from notebooks/
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC  = ROOT / "src"

# put repo + src on sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


#print("ROOT:", ROOT)
#print("SRC :", SRC, "| exists:", SRC.exists())
#print("sys.path[0:3]:", sys.path[:3])

import os
# Default to manual mode so sessions.csv NWB paths are used directly.
# Override with ACCESS_MODE=sdk to use AllenSDK auto-download.
os.environ.setdefault("ACCESS_MODE", "manual")


'manual'

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps. This can take up to a minute.

In [2]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = ROOT / "notebooks" / "00_Setup_and_Configuration.ipynb"
header  = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))

if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

All prerequisites satisfied.


## Step 1: Load configuration
We load the central config and ensure all required directories exist.

In [3]:
from config import get_config, write_config_snapshot

cfg = get_config()
cfg.ensure_dirs()
cfg

Config(access_mode='manual', pose_tool='sleap', model_name='xgboost', bin_size_s=0.025, categorical_cols=['motif_id', 'trial_type', 'stimulus_name'], mock_mode=False, outputs_dir=PosixPath('/Users/muh/projects/vbn-analysis/outputs'), cache_dir=PosixPath('/Users/muh/projects/vbn-analysis/outputs/cache'), pose_projects_dir=PosixPath('/Users/muh/projects/vbn-analysis/pose_projects'), data_dir=PosixPath('/Users/muh/projects/vbn-analysis/data'), video_source='auto', video_cache_dir=PosixPath('/Users/muh/projects/vbn-analysis/data/raw/visual-behavior-neuropixels'), video_bucket='allen-brain-observatory', video_base_path='visual-behavior-neuropixels/raw-data', video_cameras=['eye', 'face', 'side'], sessions_csv=PosixPath('/Users/muh/projects/vbn-analysis/sessions.csv'), legacy_dir=PosixPath('/Users/muh/projects/vbn-analysis/legacy'))

## Step 1b: Adjust configuration (optional)
Set access mode, pose tool, and mock mode here before running the pipeline.

In [4]:
# Optional overrides
cfg.access_mode = cfg.access_mode  # "sdk" or "manual"
cfg.pose_tool = cfg.pose_tool      # "sleap" or "dlc"
cfg.bin_size_s = cfg.bin_size_s    # e.g., 0.025
cfg.mock_mode = cfg.mock_mode      # True to force mock data
cfg

Config(access_mode='manual', pose_tool='sleap', model_name='xgboost', bin_size_s=0.025, categorical_cols=['motif_id', 'trial_type', 'stimulus_name'], mock_mode=False, outputs_dir=PosixPath('/Users/muh/projects/vbn-analysis/outputs'), cache_dir=PosixPath('/Users/muh/projects/vbn-analysis/outputs/cache'), pose_projects_dir=PosixPath('/Users/muh/projects/vbn-analysis/pose_projects'), data_dir=PosixPath('/Users/muh/projects/vbn-analysis/data'), video_source='auto', video_cache_dir=PosixPath('/Users/muh/projects/vbn-analysis/data/raw/visual-behavior-neuropixels'), video_bucket='allen-brain-observatory', video_base_path='visual-behavior-neuropixels/raw-data', video_cameras=['eye', 'face', 'side'], sessions_csv=PosixPath('/Users/muh/projects/vbn-analysis/sessions.csv'), legacy_dir=PosixPath('/Users/muh/projects/vbn-analysis/legacy'))

## Step 2: Validate dependencies
We check for core dependencies and external tools like ffmpeg.

In [5]:
import importlib.util
import shutil

deps = ["allensdk", "pynwb", "numpy", "pandas", "xgboost"]
status = {d: importlib.util.find_spec(d) is not None for d in deps}
ffmpeg = shutil.which("ffmpeg")
status, {"ffmpeg": ffmpeg}

({'allensdk': True,
  'pynwb': True,
  'numpy': True,
  'pandas': True,
  'xgboost': True},
 {'ffmpeg': '/opt/homebrew/bin/ffmpeg'})

## Step 3: Write configuration snapshot
This snapshot makes the run reproducible and documents environment settings.

In [6]:
path = write_config_snapshot()
path

PosixPath('/Users/muh/projects/vbn-analysis/outputs/reports/config_snapshot.json')